# Diagnóstico de dados e protocolo de validação — FOG / Whirlpool (G03)

**Para que serve este notebook.** Antes de gerar qualquer dado novo (sintético, FEM recriado, o que for),
precisamos responder uma pergunta: *o modelo atual está limitado por falta de amostras ou por falta de
informação nas features?* São problemas diferentes e têm soluções opostas. Gerar dados resolve o primeiro
e não faz nada pelo segundo.

Este notebook mede isso. Ele roda em cima de `../data/dataset_pre_processado.csv` e não altera nenhum arquivo.

**Ordem das seções**

| # | Seção | Pergunta que responde |
|---|---|---|
| 1 | Estrutura dos dados | Quantas amostras independentes existem de verdade? |
| 2 | `GroupKFold` — o conceito | Por que a validação atual superestima o modelo |
| 3 | Medindo o vazamento | Quanto a métrica cai quando validamos direito |
| 4 | Configurações distintas | Quantos pontos de projeto o modelo consegue diferenciar |
| 5 | Piso de ruído | Qual o melhor RMSE fisicamente possível com estas features |
| 6 | A coluna `reason` | Onde está a informação que falta |
| 7 | Extração de fatores | Quanto se ganha codificando ela |
| 8 | Harness de avaliação | Como testar dado sintético quando ele existir |

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, re
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, GroupKFold, LeaveOneGroupOut, cross_val_predict
from sklearn.metrics import mean_absolute_error, r2_score

RANDOM_STATE = 0
N_TREES = 150

df = pd.read_csv('../data/dataset_pre_processado.csv')
df.columns = [c.strip().replace('﻿', '') for c in df.columns]

FATORES = ['blower_direction', 'housing', 'board_position', 'run_case',
           'losses', 'deflector_hole', 'material_coil_support']
ALVO = 'temp'
GRUPO = 'run_id'

print('linhas:', len(df))
print('runs  :', df[GRUPO].nunique())
print('TCs   :', df['tc_number'].nunique())
df.head(3)

## 1. Estrutura dos dados: 4.065 linhas, mas não 4.065 amostras

O arquivo tem ~4.065 linhas. Isso **não** significa 4.065 observações independentes.

Cada `run_id` é **uma** execução do experimento, com **uma** configuração de hardware. Dentro dela,
cada linha é um termopar diferente medindo um ponto diferente do mesmo forno, no mesmo instante,
sob a mesma configuração.

Ou seja: as ~73 linhas de uma run compartilham todas as variáveis de entrada. Elas variam só em
*onde* o termopar está. São medições correlacionadas do mesmo evento, não repetições do experimento.

A unidade estatística real é a **run**. `n = 56`.

In [ ]:
por_run = df.groupby(GRUPO).size()
print('linhas por run — min %d, mediana %d, max %d' % (por_run.min(), por_run.median(), por_run.max()))

# Prova de que os fatores são constantes dentro de cada run:
variacao = df.groupby(GRUPO)[FATORES].nunique().max()
print()
print('valores distintos de cada fator DENTRO de uma mesma run (1 = constante):')
print(variacao.to_string())

## 2. O conceito: o que é `GroupKFold` e por que ele importa aqui

### Validação cruzada, em uma frase

Para estimar o erro de um modelo em dados que ele nunca viu, separamos parte dos dados,
treinamos no resto, e medimos o erro na parte separada. A validação cruzada (*cross-validation*)
repete isso k vezes, rodando a parte separada por todo o dataset, e junta os erros.

### `KFold` — o jeito padrão, e por que ele quebra aqui

`KFold(shuffle=True)` embaralha as **linhas** e corta em k pedaços.

Com os nossos dados, isso significa que a Run 27 pode ter 60 linhas no treino e 13 no teste.
O modelo vê, durante o treino, o resultado de 60 termopares da Run 27 — e depois é testado
em outros 13 termopares *da mesma run*, com exatamente a mesma configuração de entrada.

Ele não precisa ter aprendido física nenhuma para acertar. Basta ter memorizado
"na Run 27 as coisas ficaram quentes". Isso é **vazamento** (*data leakage*): informação do
conjunto de teste entrou no treino por uma porta lateral.

A métrica que sai daí responde à pergunta errada — *"o modelo interpola entre termopares de uma
run que ele já viu?"* — quando a pergunta do projeto é *"o modelo prevê uma configuração nova?"*.

### `GroupKFold` — o conserto

`GroupKFold` recebe um vetor `groups` e garante que **todas as linhas de um mesmo grupo caem
sempre do mesmo lado do corte**. Passando `groups = run_id`, uma run está inteira no treino
ou inteira no teste — nunca dividida.

Agora o teste simula o cenário real: chega uma configuração de hardware que nunca foi testada,
e o modelo precisa prever a temperatura em cada ponto.

### `LeaveOneGroupOut` — a versão extrema

Caso particular com k = número de grupos: treina em 55 runs, testa em 1, repete 56 vezes.
É o mais honesto e o mais caro. Com n=56 vale a pena, e é o que deve ir para a WAD.

### Resumindo

| | O que embaralha | O que a métrica mede |
|---|---|---|
| `KFold(shuffle=True)` | linhas soltas | interpolação dentro de runs conhecidas (**otimista**) |
| `GroupKFold(groups=run_id)` | runs inteiras | generalização para configuração nova (**honesto**) |
| `LeaveOneGroupOut` | uma run por vez | o mesmo, com o máximo de dados de treino |

### 2.1 Visualizando a diferença

O gráfico abaixo não é decorativo: ele mostra, para as 10 primeiras runs, quais linhas o
`KFold` manda para o teste (espalhadas por todas as runs) versus o `GroupKFold`
(runs inteiras, blocos sólidos).

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

sub = df[df[GRUPO].isin(df[GRUPO].unique()[:10])].reset_index(drop=True)
g_sub = sub[GRUPO].values
idx = np.arange(len(sub))

fold_kf = np.zeros(len(sub), int)
for i, (_, te) in enumerate(KFold(5, shuffle=True, random_state=0).split(sub)):
    fold_kf[te] = i
fold_gk = np.zeros(len(sub), int)
for i, (_, te) in enumerate(GroupKFold(5).split(sub, groups=g_sub)):
    fold_gk[te] = i

fig, axes = plt.subplots(2, 1, figsize=(11, 3.2), sharex=True)
for ax, fold, titulo in zip(axes, [fold_kf, fold_gk],
                            ['KFold(shuffle=True) — folds picotam as runs',
                             'GroupKFold(groups=run_id) — cada run inteira em um fold']):
    ax.scatter(idx, np.zeros(len(sub)), c=fold, cmap='tab10', marker='|', s=400, linewidths=2)
    ax.set_yticks([]); ax.set_title(titulo, fontsize=10, loc='left')
for r in sub[GRUPO].unique():
    pos = idx[sub[GRUPO] == r]
    for ax in axes:
        ax.axvline(pos[0] - 0.5, color='k', lw=0.6, alpha=0.4)
axes[1].set_xlabel('linha (linhas verticais pretas = fronteira entre runs)')
plt.tight_layout(); plt.show()

## 3. Medindo o vazamento

Mesmo modelo, mesmas features, só muda o esquema de validação.

In [ ]:
def montar_X(dados, extras=None):
    X = pd.get_dummies(dados[FATORES + ['key_area', 'tc_name']].astype(str))
    X['active_limit'] = dados['active_limit'].values
    X['tc_number'] = dados['tc_number'].values
    if extras is not None:
        X = pd.concat([X.reset_index(drop=True), extras.reset_index(drop=True)], axis=1)
    return X.astype(float)

def avaliar(X, y, cv, groups=None, nome=''):
    modelo = RandomForestRegressor(n_estimators=N_TREES, random_state=RANDOM_STATE, n_jobs=-1)
    pred = cross_val_predict(modelo, X.values, y, cv=cv, groups=groups, n_jobs=1)
    rmse = float(np.sqrt(((pred - y) ** 2).mean()))
    return {'esquema': nome, 'RMSE': rmse, 'MAE': mean_absolute_error(y, pred), 'R2': r2_score(y, pred)}

X = montar_X(df)
y = df[ALVO].values
g = df[GRUPO].values

resultados = [
    avaliar(X, y, KFold(5, shuffle=True, random_state=RANDOM_STATE), None, 'KFold aleatorio (VAZADO)'),
    avaliar(X, y, GroupKFold(5), g, 'GroupKFold por run_id'),
    avaliar(X, y, LeaveOneGroupOut(), g, 'LeaveOneRunOut'),
]

media_tc = df.groupby('tc_number')[ALVO].transform('mean').values
resultados.append({'esquema': 'baseline: media do TC', 'RMSE': float(np.sqrt(((media_tc - y) ** 2).mean())),
                   'MAE': mean_absolute_error(y, media_tc), 'R2': r2_score(y, media_tc)})

tabela = pd.DataFrame(resultados).set_index('esquema').round(3)
tabela

**Como ler.** A diferença entre a linha 1 e a linha 2 é o tamanho do vazamento. Se ela for grande,
a métrica reportada até agora estava inflada e o número honesto é o do `GroupKFold`.

A quarta linha é o piso de comparação: um "modelo" que ignora toda a configuração e chuta a média
histórica daquele termopar. Qualquer modelo precisa bater isso com folga para estar aprendendo
alguma coisa sobre o hardware, e não só decorando quais sensores são quentes.

**A partir daqui, use `GroupKFold` ou `LeaveOneGroupOut` em tudo.** Nunca mais `KFold` simples
neste dataset.

## 4. Quantos pontos de projeto distintos existem de verdade

Temos 56 runs. Mas o modelo só enxerga o que está nas colunas de entrada. Se duas runs têm
**o mesmo vetor de features**, elas são a mesma coisa do ponto de vista do modelo — mesmo que
no laboratório fossem experimentos diferentes.

Vamos contar quantas combinações de fatores realmente distintas existem.

In [ ]:
por_run_fatores = df.groupby(GRUPO)[FATORES].first()
chave = por_run_fatores.astype(str).agg(' | '.join, axis=1)
df['combo'] = df[GRUPO].map(chave)

n_combos = chave.nunique()
print('runs                        : %d' % len(por_run_fatores))
print('combinacoes distintas       : %d' % n_combos)
print('runs invisiveis ao modelo   : %d' % (len(por_run_fatores) - n_combos))
print()
espaco = int(np.prod([por_run_fatores[c].nunique() for c in FATORES]))
print('espaco fatorial completo    : %d combinacoes teoricas' % espaco)
print('cobertura                   : %.1f%%' % (100 * n_combos / espaco))
print()
print('--- combos com mais de uma run (mesma entrada, saidas diferentes) ---')
rep = chave.value_counts()
print(rep[rep > 1].to_string())

Runs que compartilham combinação mas produziram temperaturas diferentes são o sintoma central
deste dataset. Ou (a) existe ruído de medição, ou (b) existe um fator físico que mudou entre elas
e **não está codificado em nenhuma coluna**. A seção 6 mostra que é majoritariamente (b).

## 5. Piso de ruído: o melhor RMSE que qualquer modelo pode atingir com estas features

Esta é a conta mais importante do notebook.

Pegue todos os pares (combinação de fatores, termopar) que aparecem em mais de uma run.
Nesses pares, a entrada do modelo é **idêntica** — então a previsão dele também será idêntica.
Mas a temperatura observada não é.

Essa discordância é erro que nenhum modelo consegue eliminar, porque não há informação nas
features para distinguir os casos. Ela é o **piso de RMSE**.

Se o modelo já está perto desse piso, mais dados não ajudam. Só features novas ajudam.

In [ ]:
grupos_rep = df.groupby(['combo', 'tc_number'])[ALVO]
estat = grupos_rep.agg(['count', 'mean', 'std'])
estat = estat[estat['count'] > 1]

piso = float(np.sqrt((estat['std'] ** 2).mean()))
rmse_honesto = tabela.loc['GroupKFold por run_id', 'RMSE']

print('pares (combo, TC) replicados : %d' % len(estat))
print('desvio dentro das replicas   : mediana %.2f C | media %.2f C | p90 %.2f C | max %.2f C'
      % (estat['std'].median(), estat['std'].mean(), estat['std'].quantile(0.9), estat['std'].max()))
print()
print('PISO DE RMSE (irredutivel)   : %.2f C' % piso)
print('RMSE do modelo (GroupKFold)  : %.2f C' % rmse_honesto)
print('margem de melhoria restante  : %.2f C  (%.0f%% do erro atual)'
      % (rmse_honesto - piso, 100 * (rmse_honesto - piso) / rmse_honesto))
print()
print('amplitude do alvo: %.1f C   |   desvio-padrao do alvo: %.1f C'
      % (df[ALVO].max() - df[ALVO].min(), df[ALVO].std()))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.hist(estat['std'].dropna(), bins=40, color='#7c8fa8', edgecolor='white')
ax.axvline(piso, color='#c0392b', lw=2, label='piso RMSE = %.1f C' % piso)
ax.axvline(rmse_honesto, color='#1f6f4a', lw=2, ls='--', label='RMSE do modelo = %.1f C' % rmse_honesto)
ax.set_xlabel('desvio-padrao (C) entre runs com features identicas, mesmo termopar')
ax.set_ylabel('nº de pares')
ax.set_title('Discordancia entre replicas — o que nenhum modelo consegue explicar', fontsize=10, loc='left')
ax.legend(); plt.tight_layout(); plt.show()

### Como interpretar este resultado

- **Modelo longe do piso** → há sinal nas features que o modelo ainda não extraiu.
  Caminho: modelo melhor, mais features derivadas, adimensionalização, mais amostras.
- **Modelo colado no piso** → o modelo já extraiu tudo que existe. Gerar mais dados com as
  mesmas variáveis não move a métrica em nada. Caminho: **medir/codificar variáveis novas**.

Este é o resultado que decide a estratégia de geração de dados do projeto.

## 6. Onde está a informação que falta: a coluna `reason`

A coluna `reason` é texto livre escrito por quem rodou os ensaios. Ela descreve exatamente
o que mudou de uma run para outra — e boa parte disso é física que não virou coluna.

Vamos olhar os combos com mais de uma run e ver se as `reason` delas são iguais.

In [ ]:
diversidade = df.groupby('combo')['reason'].nunique().sort_values(ascending=False)
print('combos cujas runs tem justificativas DIFERENTES (= fator oculto):')
print(diversidade[diversidade > 1].to_string())
print()
for combo in diversidade[diversidade > 1].head(4).index:
    print('--- combo: %s' % combo[:80])
    for r in pd.unique(df.loc[df['combo'] == combo, 'reason']):
        print('      %s' % r)
    print()

Se aparecerem coisas como velocidade do soprador (*On / Medium / High*), presença do cooktop
(*wo cooktop*), variante de duto, posição do defletor ou vazão em CFM — essas são variáveis de
projeto reais. Elas mudam a temperatura e o modelo não tem acesso a nenhuma delas.

**Esse é o gargalo.** Não é o número de runs.

## 7. Extraindo os fatores escondidos

A extração abaixo é por palavra-chave e é deliberadamente simples. Ela serve para **demonstrar
o efeito**, não para ser a versão final.

> **Tarefa para o grupo:** a versão definitiva não sai de regex. Ela sai de sentar com o Felipe
> Tarallo com a lista das 56 runs e preencher, para cada uma, as colunas que faltam
> (RPM/CFM do soprador, cooktop on/off, variante de duto, posição do defletor, etc.).
> Isso é levantamento de dado, não engenharia de feature.

In [ ]:
R = df['reason'].fillna('').str.lower()

extras = pd.DataFrame({
    'f_cover':        R.str.contains('cover').astype(int),
    'f_sem_cooktop':  R.str.contains('wo cookitop|wo cooktop').astype(int),
    'f_deflector':    R.str.contains('deflector').astype(int),
    'f_duct':         R.str.contains('duct').astype(int),
    'f_tray':         R.str.contains('tray').astype(int),
    'f_hole':         R.str.contains('hole').astype(int),
    'f_plate':        R.str.contains('plate').astype(int),
    'f_spacer':       R.str.contains('spacer').astype(int),
    'f_cfm':          R.str.extract(r'(\d+)\s*cfm')[0].astype(float).fillna(-1),
    'f_rpm':          R.str.extract(r'(\d+)\s*rpm')[0].astype(float).fillna(-1),
    'f_fan_low':      R.str.contains('fan.*low|fans in low').astype(int),
    'f_fan_nivel':    R.str.contains('medium').astype(int) * 1 + R.str.contains('high').astype(int) * 2,
    'f_baseline':     R.str.contains('baseline').astype(int),
    'f_posicao':      R.str.contains('position').astype(int),
})

chave_nova = pd.concat([chave.rename('base'),
                        extras.groupby(df[GRUPO]).first().astype(str).agg('|'.join, axis=1).rename('extra')],
                       axis=1).astype(str).agg('|'.join, axis=1)

print('combinacoes distintas ANTES : %d / 56' % chave.nunique())
print('combinacoes distintas DEPOIS: %d / 56' % chave_nova.nunique())
por_run_extras = extras.groupby(df[GRUPO]).first()
resumo = pd.DataFrame({
    'runs afetadas': [(por_run_extras[c] > 0).sum() for c in por_run_extras.columns],
    'valores distintos': [por_run_extras[c].nunique() for c in por_run_extras.columns],
}, index=por_run_extras.columns)
resumo

In [ ]:
X2 = montar_X(df, extras)
linha_nova = avaliar(X2, y, GroupKFold(5), g, 'GroupKFold + fatores de reason')

comp = pd.DataFrame([tabela.loc['GroupKFold por run_id'].to_dict() | {'esquema': 'GroupKFold (features atuais)'},
                     linha_nova]).set_index('esquema').round(3)
comp['ganho RMSE %'] = (100 * (comp['RMSE'].iloc[0] - comp['RMSE']) / comp['RMSE'].iloc[0]).round(1)
comp

O ganho aqui tende a ser modesto — 14 features binárias extraídas de texto, sustentadas por 56 runs,
é pouco para uma Random Forest explorar. O ponto não é a magnitude: é a **direção**. Informação
que estava inacessível virou acessível sem nenhum dado novo, sem licença de software e sem simulação.

Com as variáveis reais preenchidas à mão (não por regex), o ganho é maior.

## 8. Harness de avaliação para quando houver dado sintético

Esta seção não roda nada agora — é a infraestrutura para a próxima etapa do projeto.

Quando existir um gerador de dados (FEM recriado, rede RC, co-kriging, o que for), a pergunta
"esse dado ajudou?" só tem uma resposta válida: **treinar com e sem ele, testar sempre e somente
em dado real, com as runs de teste fora do treino.**

Três regras inegociáveis:

1. **Teste 100% real.** Avaliar em dado sintético mede a fidelidade do gerador ao próprio gerador.
2. **Ablação.** só-real vs. real+sintético, no mesmo conjunto de teste, mesmas sementes.
3. **Sanidade física antes da métrica.** Dado sintético que viola física é lixo mesmo com bom RMSE.

In [ ]:
def ablacao(df_real, df_sintetico=None, n_splits=5, random_state=RANDOM_STATE):
    """
    Treina com e sem dado sintetico; testa SEMPRE em runs reais mantidas fora do treino.

    df_real       : DataFrame real, com as colunas FATORES + key_area/tc_name/active_limit/tc_number/temp/run_id
    df_sintetico  : DataFrame no mesmo formato, ou None. Suas linhas entram APENAS no treino.
    Retorna       : DataFrame com RMSE/MAE/R2 das duas condicoes.
    """
    y_r = df_real[ALVO].values
    g_r = df_real[GRUPO].values
    X_r = montar_X(df_real)

    if df_sintetico is not None:
        X_s = montar_X(df_sintetico)
        X_r, X_s = X_r.align(X_s, join='outer', axis=1, fill_value=0.0)
        y_s = df_sintetico[ALVO].values

    saida = []
    for rotulo, usar_sint in [('so real', False), ('real + sintetico', True)]:
        if usar_sint and df_sintetico is None:
            continue
        preds = np.zeros(len(df_real))
        for tr, te in GroupKFold(n_splits).split(X_r, y_r, groups=g_r):
            Xtr, ytr = X_r.iloc[tr], y_r[tr]
            if usar_sint:
                Xtr = pd.concat([Xtr, X_s], ignore_index=True)
                ytr = np.concatenate([ytr, y_s])
            m = RandomForestRegressor(n_estimators=N_TREES, random_state=random_state, n_jobs=-1)
            m.fit(Xtr.values, ytr)
            preds[te] = m.predict(X_r.iloc[te].values)
        saida.append({'treino': rotulo,
                      'RMSE (teste real)': float(np.sqrt(((preds - y_r) ** 2).mean())),
                      'MAE': mean_absolute_error(y_r, preds),
                      'R2': r2_score(y_r, preds)})
    return pd.DataFrame(saida).set_index('treino').round(3)


def sanidade_fisica(dados, t_ambiente=25.0):
    """
    Checagens que qualquer dado (real ou sintetico) precisa passar.
    Retorna DataFrame com o resultado de cada teste.
    """
    testes = []
    testes.append(('temperatura >= ambiente',
                   bool((dados[ALVO] >= t_ambiente).all()),
                   'min = %.1f C' % dados[ALVO].min()))
    testes.append(('temperatura dentro da faixa observada (27-274 C)',
                   bool(dados[ALVO].between(20, 320).all()),
                   'faixa = %.1f a %.1f C' % (dados[ALVO].min(), dados[ALVO].max())))
    ref = df.groupby('key_area')[ALVO].median().sort_values()
    novo = dados.groupby('key_area')[ALVO].median().reindex(ref.index)
    ordem_ok = bool(novo.dropna().is_monotonic_increasing) if novo.notna().sum() > 1 else True
    testes.append(('ordenacao relativa entre key_area preservada', ordem_ok,
                   ' < '.join(ref.index.tolist())))
    lim = dados.groupby('tc_number')[ALVO].std()
    testes.append(('nenhum TC com dispersao absurda (sd < 60 C)',
                   bool((lim.dropna() < 60).all()),
                   'sd max = %.1f C' % lim.max()))
    return pd.DataFrame(testes, columns=['teste', 'passou', 'detalhe'])


print('Funcoes prontas: ablacao(df_real, df_sintetico) e sanidade_fisica(dados)')
print()
print('Sanidade rodada sobre o proprio dado real (baseline de referencia):')
sanidade_fisica(df)

In [ ]:
print('=' * 62)
print('RESUMO — numeros para levar para a WAD e para a reuniao')
print('=' * 62)
print('RMSE KFold aleatorio (vazado, NAO usar) : %.2f C' % tabela.loc['KFold aleatorio (VAZADO)', 'RMSE'])
print('RMSE GroupKFold (honesto)              : %.2f C' % tabela.loc['GroupKFold por run_id', 'RMSE'])
print('RMSE LeaveOneRunOut (honesto, k=56)    : %.2f C' % tabela.loc['LeaveOneRunOut', 'RMSE'])
print('Piso irredutivel medido nas replicas   : %.2f C' % piso)
print('Margem restante                        : %.2f C (%.0f%% do erro atual)'
      % (rmse_honesto - piso, 100 * (rmse_honesto - piso) / rmse_honesto))
print()
print('Runs                                   : %d' % df[GRUPO].nunique())
print('Configuracoes distintas (features hoje): %d' % chave.nunique())
print('Configuracoes apos extrair reason      : %d' % chave_nova.nunique())
print('Runs indistinguiveis hoje              : %d' % (df[GRUPO].nunique() - chave.nunique()))
print('=' * 62)

## 9. Conclusão — o que isso significa para a etapa de geração de dados

Preencha com os números que saíram acima:

Os números estão impressos na célula acima. Os três que importam:

- **RMSE honesto** (`GroupKFold` / `LeaveOneRunOut`) — o erro real do modelo
- **Piso irredutível** — o melhor que qualquer modelo pode fazer com estas features
- **Configurações distintas** — quantos pontos de projeto o modelo consegue diferenciar

**Se o RMSE honesto estiver próximo do piso**, a conclusão é que o dataset não está limitado por
número de amostras, e sim por variáveis não registradas. Nesse cenário:

| Rota | Veredito |
|---|---|
| Codificar os fatores de `reason` com o parceiro | **Fazer primeiro.** Custo baixo, é a única que ataca o gargalo real |
| Adimensionalização (Buckingham Pi) | Fazer em seguida — reorganiza o espaço de entrada |
| Rede térmica RC calibrada | Vale, gera física nova de verdade |
| Recriar o FEM | Adiar. Caro, e se o dado original for de bancada não reproduz a mesma grandeza |
| CTGAN / TVAE / SMOTER | Descartar. Não inventam variável que não foi medida |

**Pendências a levar para o Tarallo:**

1. As 56 runs são simulação ou ensaio de bancada? (os nomes dos TCs sugerem bancada)
2. Existe planilha com os parâmetros que ficaram só na coluna `reason`?
3. Qual a tolerância de erro aceitável em °C? Se for ±3–5 °C, comparar com o piso medido aqui
   antes de prometer qualquer número.